In [4]:
# @title
# ============================================================
# INFRAPULSE — CELL 1
# LOAD TRAINED MOBILENETV3-SMALL
# ============================================================

import os
import time
import torch
import torch.nn as nn

from torchvision import models


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_PATH = "/content/mobilenet_v3_small_best.pth"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 80)
print("INFRAPULSE — MODEL LOADING")
print("=" * 80)

print(f"Device     : {DEVICE}")
print(f"Model path : {MODEL_PATH}")


# ============================================================
# CLASS DEFINITIONS
# ============================================================

CLASS_NAMES = [
    "Cracked_Tiles",
    "Peeling",
    "Spalling",
    "Stagnant_Water"
]

NUM_CLASSES = len(CLASS_NAMES)


# ============================================================
# DEPARTMENT MAPPING
# ============================================================

DEFECT_DEPARTMENT = {

    "Cracked_Tiles":
        "Performance",

    "Peeling":
        "Performance",

    "Spalling":
        "Structural",

    "Stagnant_Water":
        "Functional"
}


# ============================================================
# CHECK MODEL FILE
# ============================================================

if not os.path.exists(MODEL_PATH):

    raise FileNotFoundError(
        f"Model not found:\n{MODEL_PATH}"
    )


# ============================================================
# CREATE MODEL ARCHITECTURE
# ============================================================

model = models.mobilenet_v3_small(
    weights=None
)


# Replace classifier for 4 classes

model.classifier[-1] = nn.Linear(
    model.classifier[-1].in_features,
    NUM_CLASSES
)


# ============================================================
# LOAD WEIGHTS
# ============================================================

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE
)


# Handle both common saving formats

if isinstance(checkpoint, dict):

    if "model_state_dict" in checkpoint:

        state_dict = checkpoint[
            "model_state_dict"
        ]

    elif "state_dict" in checkpoint:

        state_dict = checkpoint[
            "state_dict"
        ]

    else:

        state_dict = checkpoint

else:

    state_dict = checkpoint


# Remove possible "module." prefix

clean_state_dict = {}

for key, value in state_dict.items():

    new_key = key

    if new_key.startswith("module."):

        new_key = new_key[
            len("module."):
        ]

    clean_state_dict[
        new_key
    ] = value


model.load_state_dict(
    clean_state_dict,
    strict=True
)


# ============================================================
# EVALUATION MODE
# ============================================================

model = model.to(
    DEVICE
)

model.eval()


print("\n✅ MobileNetV3-Small loaded successfully.")
print(f"Classes: {CLASS_NAMES}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

INFRAPULSE — MODEL LOADING
Device     : cpu
Model path : /content/mobilenet_v3_small_best.pth

✅ MobileNetV3-Small loaded successfully.
Classes: ['Cracked_Tiles', 'Peeling', 'Spalling', 'Stagnant_Water']
Parameters: 1,521,956


In [5]:
# @title
# ============================================================
# INFRAPULSE — CELL 2
# IMAGE CLASSIFICATION / INFERENCE
# ============================================================

import torch
from PIL import Image
from torchvision import transforms


# ============================================================
# IMAGE TRANSFORMATION
# ============================================================

inference_transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ============================================================
# PREDICTION FUNCTION
# ============================================================

def predict_image(
    image_path
):

    start_time = time.perf_counter()

    # --------------------------------------------------------
    # Load image
    # --------------------------------------------------------

    image = Image.open(
        image_path
    ).convert("RGB")

    # --------------------------------------------------------
    # Transform
    # --------------------------------------------------------

    tensor = inference_transform(
        image
    )

    tensor = tensor.unsqueeze(
        0
    ).to(DEVICE)

    # --------------------------------------------------------
    # Model inference
    # --------------------------------------------------------

    with torch.no_grad():

        logits = model(
            tensor
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predicted_index = torch.argmax(
            probabilities,
            dim=1
        ).item()

        confidence = probabilities[
            0,
            predicted_index
        ].item()

    # --------------------------------------------------------
    # Inference time
    # --------------------------------------------------------

    inference_time_ms = (
        time.perf_counter()
        -
        start_time
    ) * 1000

    predicted_class = CLASS_NAMES[
        predicted_index
    ]

    return {

        "class":
            predicted_class,

        "class_index":
            predicted_index,

        "confidence":
            confidence,

        "inference_time_ms":
            inference_time_ms
    }


print("=" * 80)
print("✅ IMAGE INFERENCE FUNCTION READY")
print("=" * 80)

✅ IMAGE INFERENCE FUNCTION READY


In [6]:
# @title
# ============================================================
# INFRAPULSE — CELL 3
# GRAD-CAM + VISUAL EXTENT ESTIMATION
# ============================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F


# ============================================================
# TARGET LAYER
# ============================================================

target_layer = model.features[-1]


gradcam_activations = None
gradcam_gradients = None


# ============================================================
# FORWARD HOOK
# ============================================================

def gradcam_forward_hook(
    module,
    input,
    output
):

    global gradcam_activations

    gradcam_activations = output


# ============================================================
# BACKWARD HOOK
# ============================================================

def gradcam_backward_hook(
    module,
    grad_input,
    grad_output
):

    global gradcam_gradients

    gradcam_gradients = grad_output[0]


# Register hooks

forward_handle = target_layer.register_forward_hook(
    gradcam_forward_hook
)

backward_handle = target_layer.register_full_backward_hook(
    gradcam_backward_hook
)


# ============================================================
# GENERATE GRAD-CAM
# ============================================================

def generate_gradcam(
    image_path
):

    global gradcam_activations
    global gradcam_gradients

    gradcam_activations = None
    gradcam_gradients = None

    # --------------------------------------------------------
    # Load image
    # --------------------------------------------------------

    image = Image.open(
        image_path
    ).convert("RGB")

    original_np = np.array(
        image
    )

    # --------------------------------------------------------
    # Transform
    # --------------------------------------------------------

    tensor = inference_transform(
        image
    )

    tensor = tensor.unsqueeze(
        0
    ).to(DEVICE)

    # --------------------------------------------------------
    # Forward
    # --------------------------------------------------------

    model.zero_grad(
        set_to_none=True
    )

    output = model(
        tensor
    )

    predicted_index = torch.argmax(
        output,
        dim=1
    ).item()

    target_score = output[
        0,
        predicted_index
    ]

    # --------------------------------------------------------
    # Backward
    # --------------------------------------------------------

    target_score.backward()

    # --------------------------------------------------------
    # Extract activations + gradients
    # --------------------------------------------------------

    activations = (
        gradcam_activations
        .detach()
        .cpu()
    )

    gradients = (
        gradcam_gradients
        .detach()
        .cpu()
    )

    # --------------------------------------------------------
    # Gradient weighting
    # --------------------------------------------------------

    weights = gradients.mean(
        dim=(2, 3),
        keepdim=True
    )

    cam = (
        weights * activations
    ).sum(
        dim=1
    )

    cam = F.relu(
        cam
    )

    cam = cam[
        0
    ].numpy()

    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    cam -= cam.min()

    if cam.max() > 0:

        cam /= cam.max()

    # --------------------------------------------------------
    # Resize CAM to original image
    # --------------------------------------------------------

    height, width = (
        original_np.shape[:2]
    )

    cam = cv2.resize(
        cam,
        (width, height)
    )

    return (
        original_np,
        cam,
        predicted_index
    )


# ============================================================
# VISIBLE EXTENT
# ============================================================

def calculate_visible_extent(
    cam,
    threshold=0.40
):

    active_region = (
        cam >= threshold
    )

    total_pixels = (
        active_region.shape[0]
        *
        active_region.shape[1]
    )

    active_pixels = (
        active_region.sum()
    )

    extent_ratio = (
        active_pixels /
        total_pixels
    )

    extent_percentage = (
        extent_ratio * 100
    )

    return (
        extent_ratio,
        extent_percentage,
        active_region
    )


# ============================================================
# EXTENT CLASS
# ============================================================

def classify_extent(
    extent_ratio
):

    if extent_ratio < 0.10:

        return (
            "SMALL",
            20
        )

    elif extent_ratio < 0.25:

        return (
            "MODERATE",
            45
        )

    elif extent_ratio < 0.45:

        return (
            "LARGE",
            70
        )

    else:

        return (
            "VERY LARGE",
            90
        )


# ============================================================
# COMPLETE VISUAL ANALYSIS
# ============================================================

def analyze_visual_extent(
    image_path,
    show_heatmap=False
):

    (
        original_image,
        cam,
        predicted_index
    ) = generate_gradcam(
        image_path
    )

    defect = CLASS_NAMES[
        predicted_index
    ]

    (
        extent_ratio,
        extent_percentage,
        active_region
    ) = calculate_visible_extent(
        cam
    )

    (
        extent_label,
        extent_score
    ) = classify_extent(
        extent_ratio
    )

    # --------------------------------------------------------
    # Optional visualization
    # --------------------------------------------------------

    if show_heatmap:

        plt.figure(
            figsize=(9, 7)
        )

        plt.imshow(
            original_image
        )

        plt.imshow(
            cam,
            alpha=0.45
        )

        plt.axis(
            "off"
        )

        plt.title(
            f"{defect} | "
            f"AI Visible Extent: "
            f"{extent_percentage:.1f}%"
        )

        plt.show()

    return {

        "visible_extent_ratio":
            extent_ratio,

        "visible_extent_percentage":
            extent_percentage,

        "extent_label":
            extent_label,

        "extent_score":
            extent_score,

        "gradcam":
            cam
    }


print("=" * 80)
print("✅ GRAD-CAM + VISUAL EXTENT ENGINE READY")
print("=" * 80)

✅ GRAD-CAM + VISUAL EXTENT ENGINE READY


In [7]:
# @title
# ============================================================
# INFRAPULSE — CELL 4
# SEVERITY + PRIORITY ENGINE
# ============================================================


# ============================================================
# DEFECT BASE RISK
# ============================================================

DEFECT_RISK_SCORE = {

    "Spalling":
        90,

    "Cracked_Tiles":
        80,

    "Stagnant_Water":
        65,

    "Peeling":
        40
}


# ============================================================
# SEVERITY ENGINE
# ============================================================

def calculate_severity(
    defect,
    extent_score
):

    defect_risk = (
        DEFECT_RISK_SCORE[
            defect
        ]
    )

    # --------------------------------------------------------
    # Severity
    #
    # 65% defect-specific risk
    # 35% visual extent
    # --------------------------------------------------------

    severity_score = (

        0.65 * defect_risk

        +

        0.35 * extent_score
    )

    if severity_score >= 75:

        severity = "HIGH"

    elif severity_score >= 50:

        severity = "MEDIUM"

    else:

        severity = "LOW"

    return (
        severity_score,
        severity
    )


# ============================================================
# FINAL PRIORITY
# ============================================================

def calculate_priority(
    defect,
    confidence,
    severity_score,
    extent_score
):

    defect_risk = (
        DEFECT_RISK_SCORE[
            defect
        ]
    )

    confidence_score = (
        confidence * 100
    )

    # --------------------------------------------------------
    # FINAL PRIORITY FORMULA
    # --------------------------------------------------------

    priority_score = (

        0.40 * defect_risk

        +

        0.30 * severity_score

        +

        0.20 * extent_score

        +

        0.10 * confidence_score
    )

    priority_score = min(
        100,
        max(
            0,
            priority_score
        )
    )

    # --------------------------------------------------------
    # Priority category
    # --------------------------------------------------------

    if priority_score >= 75:

        priority_level = "CRITICAL"

    elif priority_score >= 60:

        priority_level = "HIGH"

    elif priority_score >= 40:

        priority_level = "MEDIUM"

    else:

        priority_level = "LOW"

    return (
        priority_score,
        priority_level
    )


print("=" * 80)
print("✅ SEVERITY + PRIORITY ENGINE READY")
print("=" * 80)

print("\nPriority formula:")
print("40% × Defect Risk")
print("30% × Severity")
print("20% × Visible Extent")
print("10% × Confidence")

✅ SEVERITY + PRIORITY ENGINE READY

Priority formula:
40% × Defect Risk
30% × Severity
20% × Visible Extent
10% × Confidence


In [8]:
# @title
# ============================================================
# INFRAPULSE — CELL 5
# ISSUE DATABASE + LIVE LEADERBOARD
# ============================================================

import pandas as pd
from datetime import datetime
from IPython.display import display


# ============================================================
# DATABASE
# ============================================================

issue_database = pd.DataFrame(
    columns=[

        "Issue ID",
        "Image",
        "Defect",
        "Department",
        "Confidence",

        "Visible Extent (%)",
        "Extent",

        "Severity Score",
        "Severity",

        "Priority",
        "Priority Level",

        "Inference Time (ms)",

        "Status",
        "Timestamp"
    ]
)


# ============================================================
# ISSUE COUNTER
# ============================================================

issue_counter = 0


def generate_issue_id():

    global issue_counter

    issue_counter += 1

    return (
        f"IP-{issue_counter:04d}"
    )


# ============================================================
# CREATE LEADERBOARD
# ============================================================

def get_leaderboard():

    if len(issue_database) == 0:

        return issue_database.copy()

    # Only unresolved issues appear in
    # the active leaderboard.

    active = issue_database[
        issue_database["Status"]
        != "Resolved"
    ].copy()

    leaderboard = (
        active
        .sort_values(

            by=[
                "Priority",
                "Severity Score",
                "Confidence"
            ],

            ascending=[
                False,
                False,
                False
            ]
        )

        .reset_index(
            drop=True
        )
    )

    leaderboard.insert(
        0,
        "Rank",
        range(
            1,
            len(leaderboard) + 1
        )
    )

    return leaderboard


# ============================================================
# DISPLAY LEADERBOARD
# ============================================================

def display_leaderboard():

    leaderboard = get_leaderboard()

    print("\n")

    print("=" * 130)
    print("              🚨 INFRAPULSE — LIVE PRIORITY LEADERBOARD")
    print("=" * 130)

    print(
        f"\nActive Issues: "
        f"{len(leaderboard)}"
    )

    print(
        f"Last Updated: "
        f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    )

    print()

    if len(leaderboard) == 0:

        print(
            "No active issues."
        )

        return

    display_df = leaderboard[
        [

            "Rank",
            "Issue ID",
            "Image",
            "Defect",
            "Department",

            "Confidence",
            "Visible Extent (%)",

            "Severity",
            "Priority",
            "Priority Level",

            "Inference Time (ms)",
            "Status"
        ]
    ].copy()

    # --------------------------------------------------------
    # Formatting
    # --------------------------------------------------------

    display_df[
        "Confidence"
    ] = display_df[
        "Confidence"
    ].apply(
        lambda x:
        f"{x:.2%}"
    )

    display_df[
        "Visible Extent (%)"
    ] = display_df[
        "Visible Extent (%)"
    ].apply(
        lambda x:
        f"{x:.1f}%"
    )

    display_df[
        "Priority"
    ] = display_df[
        "Priority"
    ].apply(
        lambda x:
        f"{x:.1f}"
    )

    display_df[
        "Inference Time (ms)"
    ] = display_df[
        "Inference Time (ms)"
    ].apply(
        lambda x:
        f"{x:.2f}"
    )

    display(
        display_df.style
        .set_properties(
            **{
                "text-align":
                    "center"
            }
        )
    )


# ============================================================
# ADD ISSUE TO DATABASE
# ============================================================

def add_issue(
    image_path
):

    global issue_database

    # --------------------------------------------------------
    # Classification
    # --------------------------------------------------------

    classification = predict_image(
        image_path
    )

    defect = classification[
        "class"
    ]

    confidence = classification[
        "confidence"
    ]

    classification_time = classification[
        "inference_time_ms"
    ]

    # --------------------------------------------------------
    # Visual extent
    # --------------------------------------------------------

    visual = analyze_visual_extent(
        image_path,
        show_heatmap=False
    )

    extent_score = visual[
        "extent_score"
    ]

    # --------------------------------------------------------
    # Severity
    # --------------------------------------------------------

    (
        severity_score,
        severity
    ) = calculate_severity(

        defect,

        extent_score
    )

    # --------------------------------------------------------
    # Priority
    # --------------------------------------------------------

    (
        priority_score,
        priority_level
    ) = calculate_priority(

        defect,

        confidence,

        severity_score,

        extent_score
    )

    # --------------------------------------------------------
    # Department
    # --------------------------------------------------------

    department = (
        DEFECT_DEPARTMENT[
            defect
        ]
    )

    # --------------------------------------------------------
    # Total processing time
    # --------------------------------------------------------

    # Grad-CAM adds processing time, so measure
    # the complete pipeline separately.

    # For now classification_time is retained
    # as the actual classifier inference time.

    # --------------------------------------------------------
    # Create record
    # --------------------------------------------------------

    issue = {

        "Issue ID":
            generate_issue_id(),

        "Image":
            os.path.basename(
                image_path
            ),

        "Defect":
            defect,

        "Department":
            department,

        "Confidence":
            confidence,

        "Visible Extent (%)":
            visual[
                "visible_extent_percentage"
            ],

        "Extent":
            visual[
                "extent_label"
            ],

        "Severity Score":
            severity_score,

        "Severity":
            severity,

        "Priority":
            priority_score,

        "Priority Level":
            priority_level,

        "Inference Time (ms)":
            classification_time,

        "Status":
            "Submitted",

        "Timestamp":
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    }

    # --------------------------------------------------------
    # Append
    # --------------------------------------------------------

    issue_database.loc[
        len(issue_database)
    ] = issue

    return issue


print("=" * 80)
print("✅ ISSUE DATABASE + LIVE LEADERBOARD READY")
print("=" * 80)

✅ ISSUE DATABASE + LIVE LEADERBOARD READY


In [11]:
# @title
# ============================================================
# INFRAPULSE — CELL 6
# FINAL MULTI-IMAGE ISSUE SUBMISSION
# ============================================================

from google.colab import files
from PIL import Image
import os
import time


print("=" * 110)
print("              🏢 INFRAPULSE — NEW MAINTENANCE ISSUES")
print("=" * 110)

print(
    "\nUpload up to 20 defect images."
)

print(
    "The system will automatically:"
)

print(
    "  1. Detect the defect"
)

print(
    "  2. Assign department"
)

print(
    "  3. Estimate visible extent"
)

print(
    "  4. Estimate severity"
)

print(
    "  5. Calculate priority"
)

print(
    "  6. Add issue to database"
)

print(
    "  7. Re-rank the leaderboard"
)

print("\n")


# ============================================================
# UPLOAD
# ============================================================

uploaded = files.upload()


if len(uploaded) > 20:

    print(
        f"\n⚠️ {len(uploaded)} files uploaded."
    )

    print(
        "Only first 20 will be processed."
    )

    selected_files = list(
        uploaded.keys()
    )[:20]

else:

    selected_files = list(
        uploaded.keys()
    )


print(
    f"\nProcessing "
    f"{len(selected_files)} images..."
)


# ============================================================
# PROCESS EACH IMAGE
# ============================================================

for i, filename in enumerate(
    selected_files,
    start=1
):

    print("\n" + "-" * 110)

    print(
        f"[{i}/{len(selected_files)}] "
        f"{filename}"
    )

    try:

        # Validate image

        Image.open(
            filename
        ).convert("RGB")

        # Start complete pipeline timer

        total_start = (
            time.perf_counter()
        )

        # Add to database

        issue = add_issue(
            filename
        )

        total_time = (
            time.perf_counter()
            -
            total_start
        ) * 1000

        print(
            f"✓ Issue ID    : "
            f"{issue['Issue ID']}"
        )

        print(
            f"✓ Defect      : "
            f"{issue['Defect']}"
        )

        print(
            f"✓ Department  : "
            f"{issue['Department']}"
        )

        print(
            f"✓ Confidence  : "
            f"{issue['Confidence']:.2%}"
        )

        print(
            f"✓ Extent      : "
            f"{issue['Visible Extent (%)']:.1f}%"
            f" ({issue['Extent']})"
        )

        print(
            f"✓ Severity    : "
            f"{issue['Severity']}"
            f" ({issue['Severity Score']:.1f})"
        )

        print(
            f"✓ Priority    : "
            f"{issue['Priority']:.1f}"
            f" [{issue['Priority Level']}]"
        )

        print(
            f"✓ Classifier inference : "
            f"{issue['Inference Time (ms)']:.2f} ms"
        )

        print(
            f"✓ Total pipeline time  : "
            f"{total_time:.2f} ms"
        )

    except Exception as e:

        print(
            f"❌ Failed: {filename}"
        )

        print(
            f"Error: {e}"
        )


# ============================================================
# FINAL LEADERBOARD
# ============================================================

display_leaderboard()

              🏢 INFRAPULSE — NEW MAINTENANCE ISSUES

Upload up to 20 defect images.
The system will automatically:
  1. Detect the defect
  2. Assign department
  3. Estimate visible extent
  4. Estimate severity
  5. Calculate priority
  6. Add issue to database
  7. Re-rank the leaderboard




Saving 2aef3734-98ee-42ce-a661-f8f1d394aae7.jpg to 2aef3734-98ee-42ce-a661-f8f1d394aae7.jpg

Processing 1 images...

--------------------------------------------------------------------------------------------------------------
[1/1] 2aef3734-98ee-42ce-a661-f8f1d394aae7.jpg
✓ Issue ID    : IP-0003
✓ Defect      : Spalling
✓ Department  : Structural
✓ Confidence  : 99.58%
✓ Extent      : 16.5% (MODERATE)
✓ Severity    : MEDIUM (74.2)
✓ Priority    : 77.2 [CRITICAL]
✓ Classifier inference : 49.76 ms
✓ Total pipeline time  : 141.80 ms


              🚨 INFRAPULSE — LIVE PRIORITY LEADERBOARD

Active Issues: 3
Last Updated: 2026-08-30 19:38:24



,Rank,Issue ID,Image,Defect,Department,Confidence,Visible Extent (%),Severity,Priority,Priority Level,Inference Time (ms),Status
0,1,IP-0001,7eb87aff-ce22-41a7-9777-3e1ca3667271.jpg,Cracked_Tiles,Performance,99.93%,32.3%,HIGH,78.9,CRITICAL,252.30,Submitted
1,2,IP-0002,a9ee5885-03bf-4512-a4d0-d628a5207fa1.jpg,Cracked_Tiles,Performance,98.42%,30.6%,HIGH,78.8,CRITICAL,55.32,Submitted
2,3,IP-0003,2aef3734-98ee-42ce-a661-f8f1d394aae7.jpg,Spalling,Structural,99.58%,16.5%,MEDIUM,77.2,CRITICAL,49.76,Submitted


In [12]:
# @title
# ============================================================
# INFRAPULSE — CELL 7
# LIVE QUEUE UPDATE
# ============================================================

print("=" * 100)
print("              🔄 INFRAPULSE — NEW LIVE COMPLAINT")
print("=" * 100)

print(
    "\nUpload new complaint image(s)."
)

uploaded_live = files.upload()


for filename in uploaded_live.keys():

    print("\n" + "-" * 100)

    print(
        f"NEW COMPLAINT: {filename}"
    )

    try:

        Image.open(
            filename
        ).convert("RGB")

        start = time.perf_counter()

        issue = add_issue(
            filename
        )

        total_time = (
            time.perf_counter()
            -
            start
        ) * 1000

        print(
            f"\n✓ Complaint accepted"
        )

        print(
            f"  Issue ID   : "
            f"{issue['Issue ID']}"
        )

        print(
            f"  Defect     : "
            f"{issue['Defect']}"
        )

        print(
            f"  Department : "
            f"{issue['Department']}"
        )

        print(
            f"  Severity   : "
            f"{issue['Severity']}"
        )

        print(
            f"  Priority   : "
            f"{issue['Priority']:.1f}"
        )

        print(
            f"  Processing : "
            f"{total_time:.2f} ms"
        )

    except Exception as e:

        print(
            f"❌ Failed: {filename}"
        )

        print(
            f"Error: {e}"
        )


# ============================================================
# AUTOMATICALLY REFRESH LEADERBOARD
# ============================================================

print(
    "\n🔄 Updating priority queue..."
)

display_leaderboard()

              🔄 INFRAPULSE — NEW LIVE COMPLAINT

Upload new complaint image(s).


Saving 2b31ea2f-6040-4cd6-b4b2-538318ae0e96.jpg to 2b31ea2f-6040-4cd6-b4b2-538318ae0e96.jpg

----------------------------------------------------------------------------------------------------
NEW COMPLAINT: 2b31ea2f-6040-4cd6-b4b2-538318ae0e96.jpg

✓ Complaint accepted
  Issue ID   : IP-0004
  Defect     : Peeling
  Department : Performance
  Severity   : MEDIUM
  Priority   : 58.4
  Processing : 201.82 ms

🔄 Updating priority queue...


              🚨 INFRAPULSE — LIVE PRIORITY LEADERBOARD

Active Issues: 4
Last Updated: 2026-08-30 19:40:15



,Rank,Issue ID,Image,Defect,Department,Confidence,Visible Extent (%),Severity,Priority,Priority Level,Inference Time (ms),Status
0,1,IP-0001,7eb87aff-ce22-41a7-9777-3e1ca3667271.jpg,Cracked_Tiles,Performance,99.93%,32.3%,HIGH,78.9,CRITICAL,252.30,Submitted
1,2,IP-0002,a9ee5885-03bf-4512-a4d0-d628a5207fa1.jpg,Cracked_Tiles,Performance,98.42%,30.6%,HIGH,78.8,CRITICAL,55.32,Submitted
2,3,IP-0003,2aef3734-98ee-42ce-a661-f8f1d394aae7.jpg,Spalling,Structural,99.58%,16.5%,MEDIUM,77.2,CRITICAL,49.76,Submitted
3,4,IP-0004,2b31ea2f-6040-4cd6-b4b2-538318ae0e96.jpg,Peeling,Performance,71.52%,52.9%,MEDIUM,58.4,MEDIUM,63.14,Submitted


In [13]:
# @title
# ============================================================
# INFRAPULSE — CELL 8
# ISSUE STATUS MANAGEMENT
# ============================================================

def update_issue_status(
    issue_id,
    new_status
):

    global issue_database

    valid_statuses = [

        "Submitted",
        "Assigned",
        "In Progress",
        "Resolved"
    ]

    if new_status not in valid_statuses:

        raise ValueError(
            f"Invalid status. "
            f"Choose from: "
            f"{valid_statuses}"
        )

    mask = (
        issue_database["Issue ID"]
        ==
        issue_id
    )

    if not mask.any():

        print(
            f"❌ Issue {issue_id} not found."
        )

        return

    issue_database.loc[
        mask,
        "Status"
    ] = new_status

    print(
        f"✅ {issue_id} → "
        f"{new_status}"
    )

    # Automatically refresh queue

    display_leaderboard()


print("=" * 80)
print("✅ STATUS MANAGEMENT READY")
print("=" * 80)

✅ STATUS MANAGEMENT READY


In [14]:
# @title
# ============================================================
# INFRAPULSE — CELL 9
# SAVE ISSUE DATABASE
# ============================================================

DATABASE_PATH = (
    "/content/infrapulse_issue_database.csv"
)

issue_database.to_csv(
    DATABASE_PATH,
    index=False
)

print("=" * 80)
print("✅ INFRAPULSE DATABASE SAVED")
print("=" * 80)

print(
    f"\nPath:\n{DATABASE_PATH}"
)

print(
    f"\nTotal issues: "
    f"{len(issue_database)}"
)

✅ INFRAPULSE DATABASE SAVED

Path:
/content/infrapulse_issue_database.csv

Total issues: 4
